# Optional - export train/test data if desired as separate files 
2025-11-13, Alexander Minidis

As name implies, optional, if one prefers to deal with plain csv files and wants to do a hard set test/train split.

But since later pipeline handles everything automatic and smaller tweaks have been implemented (2026), perhaps only useful if one wants to remodel from scratch?

In [ ]:
import sys

from typing import Any
from pathlib import Path
import pandas as pd
import numpy as np

notebookdir = Path.cwd().parent
sys.path.append(str(notebookdir))  # this way we can import src modules even in different subfolders

import sqlalchemy as sa
from sqlalchemy.orm import sessionmaker
from src.db_schema import *
from src.db_utils import get_all_data
from src.rdkit_tools import MACCS_NAMES
from src.ml_tools import decorrelate, drop_irrelevant_columns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 250)

In [ ]:
# set directories and filenames, load database
working_dir = Path.cwd().parent
data_dir = working_dir / "processed_data"
database_file = data_dir / "hsbd_t_half_data.db"
engine = sa.create_engine(f"sqlite:///{database_file}")
Session = sessionmaker(bind=engine)
output_dir = data_dir / "X_y_splits"
output_dir.mkdir(exist_ok=True)

In [3]:
# get all data
air_data = get_all_data("air", Session)
soil_data = get_all_data("soil", Session)
water_data = get_all_data("water", Session)
sediment_data = get_all_data("sediment", Session)

air_data = drop_irrelevant_columns(air_data, to_drop=["None"])
water_data = drop_irrelevant_columns(water_data, to_drop=["None"])
soil_data = drop_irrelevant_columns(soil_data, to_drop=["None"])
sediment_data = drop_irrelevant_columns(sediment_data, to_drop=["None"])

target_column = "T_half_days"

In [4]:
data_to_use = air_data.copy()

## Preprocessing

In [5]:
# we will only use air data
X, y = data_to_use.drop(columns=[target_column]), data_to_use[target_column]
print(f"Number of features: {X.shape[1]}, number of samples: {X.shape[0]}")

Number of features: 380, number of samples: 523


### 2. Scaling/normalization

In [6]:
# scaling
scaler = StandardScaler()
# X_scaled without MACCS
X_MACCS = pd.DataFrame(X[MACCS_NAMES])
X_rdkit = pd.DataFrame(X.drop(columns=MACCS_NAMES))
X_scaled = pd.DataFrame(scaler.fit_transform(X_rdkit), columns=X_rdkit.columns)
X_scaled = X_scaled.reset_index(drop=True)
X_MACCS = X_MACCS.reset_index(drop=True)
X_scaled = pd.concat([X_scaled, X_MACCS], axis=1)

In [7]:
# remove zero std columns (no variance)
zero_std_cols = X_scaled.columns[X_scaled.std() == 0]
X_scaled = X_scaled.drop(columns=zero_std_cols)
print(f"Number of features: {X_scaled.shape[1]}, number of samples: {X_scaled.shape[0]}")

Number of features: 312, number of samples: 523


In [8]:
# drop columns hihgly correlated to some others
cols_to_drop = decorrelate(X_scaled, target_column, threshold=0.95)
X_decorrelated = X_scaled.drop(columns=cols_to_drop)
print(f"Number of features: {X_decorrelated.shape[1]}, number of samples: {X_decorrelated.shape[0]}")

Number of features: 253, number of samples: 523


In [9]:
X = X_decorrelated.copy()
# y = np.log10(y) # choose if log or not

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [11]:
# different output formats, e.g.:
X_train.to_csv(output_dir / "X_train.csv", index=False)
X_test.to_csv(output_dir / "X_test.csv", index=False)
y_train.to_csv(output_dir / "y_train.csv", index=False, header=True)
y_test.to_csv(output_dir / "y_test.csv", index=False, header=True)

## for testing elsewhere

In [12]:
# get all data
air_data = get_all_data("air", Session)
soil_data = get_all_data("soil", Session)
water_data = get_all_data("water", Session)
sediment_data = get_all_data("sediment", Session)

# drop all columns except T_half_days and SMILES
air_data = air_data[["Canonical_smiles", "T_half_days"]]
water_data = water_data[["Canonical_smiles", "T_half_days"]]
soil_data = soil_data[["Canonical_smiles", "T_half_days"]]
sediment_data = sediment_data[["Canonical_smiles", "T_half_days"]]

# set columns names to "smiles" and "labels"
air_data.columns = ["smiles", "labels"]
water_data.columns = ["smiles", "labels"]
soil_data.columns = ["smiles", "labels"]
sediment_data.columns = ["smiles", "labels"]

In [13]:
def split_and_export(df: pd.DataFrame, output_dir: Path, type: str) -> None:
    X, y = df["smiles"], df["labels"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    train = pd.concat([X_train.reset_index(drop=True), y_train.reset_index(drop=True)], axis=1)
    test = pd.concat([X_test.reset_index(drop=True), y_test.reset_index(drop=True)], axis=1)

    train.to_csv(output_dir / f"{type}_X_y_train.csv", index=False)
    test.to_csv(output_dir / f"{type}_X_y_test.csv", index=False)


split_and_export(air_data, output_dir, "air")
split_and_export(water_data, output_dir, "water")
split_and_export(soil_data, output_dir, "soil")
split_and_export(sediment_data, output_dir, "sediment")